In [1]:
import sys
import os
import MetaTrader5 as mt5
import time

project_dir = os.path.abspath("..")
if project_dir not in sys.path:
    sys.path.append(project_dir)

import polars as pl

from src.infra.mtBase import mtBase

In [2]:
mtb = mtBase(
    account="mt5demo_acc_usd",
    credentials_path=os.path.join("..", "secrets", "mt5_acc_cred.yaml"),
    config_path=os.path.join("..", "secrets", "mt5_config.ini"),
)
mtb.mt5_init()

MetaTrader 5 connection established


In [3]:
pred = {
    "symbol": "INTC",
    "last_training_day": "2025-10-27T00:00:00Z",
    "last_close_price": 70.17,
    "n_trading_days": 5,
    "score": 0.76,
    "sl_pct": 0.08,
    "tp_pct": 0.3
}

In [4]:
# Import required classes
from src.infra.OrderData import OrderData
from src.infra.OrderClient import OrderClient
import MetaTrader5 as mt5

In [5]:
pricetick = mtb.get_symbol_price(pred['symbol'], wait_sec=0.2)
symbol_info = mtb.get_symbol_info(pred['symbol'], wait_sec=0.2)

entry_price = pricetick['ask']
ask_price = pricetick['ask']
bid_price = pricetick['bid']
spread = ask_price - bid_price
spread_rel = spread / (entry_price + 1e-6)
print(f"Spread: {spread:.5f} USD, Spread relative: {spread_rel*100:.4f} %")

Spread: 0.05000 USD, Spread relative: 0.1248 %


In [6]:
# Calculate stop loss and take profit prices
# For buy position: SL = price * (1 - sl_pct), TP = price * (1 + tp_pct)
sl_pct = pred["sl_pct"]  # 0.07 = 7%
tp_pct = pred["tp_pct"]  # 0.9 = 90%

sl_price = entry_price * (1 - sl_pct)
tp_price = entry_price * (1 + tp_pct)

# Round to appropriate digits
digits = symbol_info['digits']
sl_price = round(sl_price, digits)
tp_price = round(tp_price, digits)

print(f"Stop Loss: {sl_price} (price * (1 - {sl_pct}) = {entry_price} * {1-sl_pct})")
print(f"Take Profit: {tp_price} (price * (1 + {tp_pct}) = {entry_price} * {1+tp_pct})")

Stop Loss: 36.85 (price * (1 - 0.08) = 40.05 * 0.92)
Take Profit: 52.06 (price * (1 + 0.3) = 40.05 * 1.3)


In [7]:
# Create OrderData from the prediction
order_data = OrderData(
    type=mt5.ORDER_TYPE_BUY,  # Buy order
    volume=1,  # 0.01 lot for testing
    price=entry_price,
    sl=sl_price,
    tp=tp_price,
    symbol=pred["symbol"],
    comment=f"Test order from prediction",
    magic=12345,  # Magic number for identification
    action=mt5.TRADE_ACTION_DEAL,
    type_time=mt5.ORDER_TIME_DAY,
    type_filling=mt5.ORDER_FILLING_FOK,
    deviation=1
)

print("Created OrderData:")
print(order_data)

Created OrderData:
OrderData(ticket=None, symbol='INTC', type=0, volume=1, price=40.05, sl=36.85, tp=52.06, magic=12345)


In [8]:
# Create OrderClient and convert OrderData to request
order_client = OrderClient(mtb)
request = order_client.to_request_dict(order_data)

print("Created MT5 request:")
for key, value in request.items():
    print(f"  {key}: {value}")

Created MT5 request:
  symbol: INTC
  volume: 1.0
  type: 0
  price: 40.05
  magic: 12345
  action: 1
  type_time: 1
  type_filling: 0
  sl: 36.85
  tp: 52.06
  comment: Test order from prediction
  deviation: 1


In [11]:
# Apply mtb.order_check(req) to validate the order
print("Performing order check...")
check_result: mt5.OrderCheckResult = mtb.order_check(request)

print(f"\nOrder check result:")
print(f"Return code: {check_result.retcode}")

if hasattr(check_result, 'comment'):
    print(f"Comment: {check_result.comment}")

if hasattr(check_result, 'margin'):
    print(f"Required margin: {check_result.margin}")

if hasattr(check_result, 'profit'):
    print(f"Expected profit: {check_result.profit}")

# Check if the order is valid
if check_result.retcode == mt5.TRADE_RETCODE_DONE or check_result.retcode == 0:
    print("✅ Order check PASSED - Order is valid!")
else:
    print(f"❌ Order check FAILED - Error code: {check_result.retcode}")

Performing order check...

Order check result:
Return code: 0
Comment: Done
Required margin: 412.69
Expected profit: 27.2
✅ Order check PASSED - Order is valid!


In [12]:
# WARNING: The following line actually places the order !!!!!!!!
mtb.place_market_order(request)

OrderSendResult(retcode=10018, deal=0, order=0, volume=0.0, price=0.0, bid=0.0, ask=0.0, comment='Market closed', request_id=3078490568, retcode_external=0, request=TradeRequest(action=1, magic=12345, order=0, symbol='INTC', volume=1.0, price=40.05, stoplimit=0.0, sl=36.85, tp=52.06, deviation=1, type=0, type_filling=0, type_time=1, expiration=0, comment='Test order from prediction', position=0, position_by=0))

In [24]:
# Use PositionClient to get positions and create cancel request
from src.infra.PositionClient import PositionClient

# Create PositionClient
pos_client = PositionClient(mtb)

# Get all positions
positions = pos_client.get_positions()
print(f"Found {len(positions)} positions")

# Get magic number from previous request
magic_number = request.get('magic', 0)
print(f"Magic number from previous request: {magic_number}")

# Get positions with our magic number
our_positions = pos_client.get_positions_by_magic(magic_number)
print(f"Found {len(our_positions)} positions with magic {magic_number}")

if our_positions:
    # Get the first position (assuming there's only one)
    position = our_positions[0]
    print(f"Position ticket: {position.ticket}, Symbol: {position.symbol}")
    
    # Create cancel request
    cancel_request = pos_client.close_position_request(
        pos = position,
        deviation_pts=10,
        magic=magic_number
    )
    
    print("Cancel request:")
    for key, value in cancel_request.items():
        print(f"  {key}: {value}")
else:
    print("No positions found with the specified magic number")

Found 30 positions
Magic number from previous request: 12345
Found 0 positions with magic 12345
No positions found with the specified magic number


In [13]:
# Apply mtb.order_check(req) to validate the order
print("Performing order check...")
check_result: mt5.OrderCheckResult = mtb.order_check(cancel_request)

print(f"\nOrder check result:")
print(f"Return code: {check_result.retcode}")

if hasattr(check_result, 'comment'):
    print(f"Comment: {check_result.comment}")

# Check if the order is valid
if check_result.retcode == mt5.TRADE_RETCODE_DONE or check_result.retcode == 0:
    print("✅ Order check PASSED - Order is valid!")
else:
    print(f"❌ Order check FAILED - Error code: {check_result.retcode}")

Performing order check...

Order check result:
Return code: 0
Comment: Done
✅ Order check PASSED - Order is valid!


In [14]:
# WARNING: The following line actually places the order !!!!!!!!
#mtb.place_market_order(cancel_request)

OrderSendResult(retcode=10009, deal=53650103884, order=53841969726, volume=1.0, price=68.43, bid=68.43, ask=68.45, comment='Request executed', request_id=1788178973, retcode_external=0, request=TradeRequest(action=1, magic=12345, order=0, symbol='KO', volume=1.0, price=0.0, stoplimit=0.0, sl=0.0, tp=0.0, deviation=10, type=1, type_filling=0, type_time=0, expiration=0, comment='Closing position 53841025227', position=53841025227, position_by=0))